In [1]:
import pennylane as qml
import numpy as np

In [2]:
n_data = 9
n_anc = 8
n_qubits = n_data + n_anc

data_qubits = list(range(n_data))
anc_z = list(range(n_data, n_data + 4))
anc_x = list(range(n_data + 4, n_qubits))

DATA_WIRES = data_qubits
ANC_WIRES = anc_z + anc_x

# 각 stabilizer가 어떤 data qubit에 연결되는지
X_stabilizers = [
    [0,1],
    [1,2,4,5],
    [3,4,6,7],
    [7,8]
]

Z_stabilizers = [
    [0,1,3,4],
    [2,5],
    [3,6],
    [4,5,7,8],
]

In [41]:
# --------------------------------------------------
# Pauli utilities
# --------------------------------------------------

def apply_pauli(pauli_type, wire):
    if pauli_type == "X":
        qml.PauliX(wires=wire)
    elif pauli_type == "Y":
        qml.PauliY(wires=wire)
    elif pauli_type == "Z":
        qml.PauliZ(wires=wire)
    else:
        raise ValueError(f"Unknown Pauli type: {pauli_type}")


def apply_pauli_error(error_types, error_wires):
    """
    error_types = ["X", "Z"]
    error_wires = [control, target]
    """
    for p, w in zip(error_types, error_wires):
        apply_pauli(p, w)

In [42]:
def get_faults_for_round(fault_schedule, current_round):
    """
    fault_schedule 예시:

    [
        {"round": 1, "wires": [0, 9], "error": ["X", "Z"]},
        {"round": 3, "wires": [4, 12], "error": ["X", "X"]},
    ]
    """
    return [
        fault for fault in fault_schedule
        if fault["round"] == current_round
    ]

def inject_faults_after_cnot(control, target, current_faults):
    """
    현재 실행된 CNOT(control, target) 직후에
    해당 위치에 지정된 fault가 있으면 Pauli error 삽입.
    """

    for fault in current_faults:
        if fault["control"] == control and fault["target"] == target:
            apply_pauli_error(
                error_types=fault["error_types"],
                error_wires=fault["error_wires"],
            )

In [44]:
# --------------------------------------------------
# Z stabilizer measurement
# --------------------------------------------------

def measure_Z_stabilizer_with_fault_schedule(
    anc,
    qubits,
    current_faults,
):
    """
    Z stabilizer measurement.

    CNOT direction:
        data q -> ancilla
    """

    for q in qubits:
        qml.CNOT(wires=[q, anc])

        inject_faults_after_cnot(
            control=q,
            target=anc,
            current_faults=current_faults,
        )

    m = qml.measure(anc, reset=True)
    return m


# --------------------------------------------------
# X stabilizer measurement
# --------------------------------------------------

def measure_X_stabilizer_with_fault_schedule(
    anc,
    qubits,
    current_faults,
):
    """
    X stabilizer measurement.

    CNOT direction:
        ancilla -> data q
    """

    qml.Hadamard(wires=anc)

    for q in qubits:
        qml.CNOT(wires=[anc, q])

        inject_faults_after_cnot(
            control=anc,
            target=q,
            current_faults=current_faults,
        )

    qml.Hadamard(wires=anc)

    m = qml.measure(anc, reset=True)
    return m

In [6]:
def prepare_logical_zero():
    """
    Prepare |0_L> of the d=3 rotated surface code
    with logical Z_L = Z0 Z1 Z2.
    """

    # Independent variables:
    # a -> qubit 0
    # b -> qubit 2
    # c -> qubit 3
    # d -> qubit 8
    qml.Hadamard(wires=0)
    qml.Hadamard(wires=2)
    qml.Hadamard(wires=3)
    qml.Hadamard(wires=8)

    # q1 = a ⊕ b
    qml.CNOT(wires=[0, 1])
    qml.CNOT(wires=[2, 1])

    # q4 = b ⊕ c
    qml.CNOT(wires=[2, 4])
    qml.CNOT(wires=[3, 4])

    # q5 = b
    qml.CNOT(wires=[2, 5])

    # q6 = c
    qml.CNOT(wires=[3, 6])

    # q7 = c ⊕ d
    qml.CNOT(wires=[3, 7])
    qml.CNOT(wires=[8, 7])

In [45]:
# --------------------------------------------------
# One syndrome extraction round
# --------------------------------------------------

def syndrome_round_with_fault_schedule(
    current_round,
    fault_schedule,
):
    """
    한 round에서 8개 stabilizer 측정.

    반환 순서:
        [Z0, Z1, Z2, Z3, X0, X1, X2, X3]
    """

    current_faults = get_faults_for_round(
        fault_schedule=fault_schedule,
        current_round=current_round,
    )

    round_record = []

    for i, stab in enumerate(Z_stabilizers):
        m = measure_Z_stabilizer_with_fault_schedule(
            anc=anc_z[i],
            qubits=stab,
            current_faults=current_faults,
        )
        round_record.append(m)

    for i, stab in enumerate(X_stabilizers):
        m = measure_X_stabilizer_with_fault_schedule(
            anc=anc_x[i],
            qubits=stab,
            current_faults=current_faults,
        )
        round_record.append(m)

    return round_record

In [46]:
# --------------------------------------------------
# Repeated syndrome QNode
# --------------------------------------------------

def make_fault_schedule_repeated_syndrome_qnode(n_rounds, shots=1):
    dev = qml.device("default.qubit", wires=n_qubits, shots=shots)

    @qml.qnode(dev)
    def repeated_syndrome_with_fault_schedule(
        fault_schedule=[],
        initial_error_list=[],
        initial_error_wires=[],
    ):
        measurement_record = []

        prepare_logical_zero()

        qml.Barrier(wires=DATA_WIRES)

        # optional initial error
        for p, w in zip(initial_error_list, initial_error_wires):
            apply_pauli(p, w)

        for t in range(n_rounds):
            round_record = syndrome_round_with_fault_schedule(
                current_round=t,
                fault_schedule=fault_schedule,
            )
            measurement_record.extend(round_record)

        return [qml.sample(m) for m in measurement_record]

    return repeated_syndrome_with_fault_schedule

In [22]:
def reshape_raw_syndrome(raw, n_rounds, shots, n_stabilizers=8):
    raw = np.array(raw)
    syndrome = raw.reshape(n_rounds, n_stabilizers, shots)
    syndrome = np.moveaxis(syndrome, 2, 0)
    return syndrome


def compute_detection_events(syndrome):
    return syndrome[:, 1:, :] ^ syndrome[:, :-1, :]


def print_single_shot_sequence(syndrome, detection_events, shot_idx=0):
    print("Syndrome sequence")
    print("-----------------")

    for t in range(syndrome.shape[1]):
        z_part = syndrome[shot_idx, t, :4]
        x_part = syndrome[shot_idx, t, 4:]

        print(f"round {t}: Z={z_part}  X={x_part}")

    print()
    print("Detection events")
    print("----------------")

    for t in range(detection_events.shape[1]):
        z_part = detection_events[shot_idx, t, :4]
        x_part = detection_events[shot_idx, t, 4:]

        print(f"between round {t} and {t+1}: Z={z_part}  X={x_part}")

In [47]:
def make_fault_schedule(controls, targets, error_wires_list, error_types_list, error_rounds):
    """
    controls: list[int]
    targets: list[int]
    error_wires_list: list[list[int]]
    error_types_list: list[list[str]]
    error_rounds: list[int]

    example:
        controls = [0, 0, 0, 0]
        targets = [9, 9, 9, 9]
        error_wires_list = [[0, 9]] * 4
        error_types_list = [["X", "Z"]] * 4
        error_rounds = [0, 1, 2, 3]
    """

    fault_schedule = []

    for c, t, ew, et, r in zip(
        controls,
        targets,
        error_wires_list,
        error_types_list,
        error_rounds,
    ):
        fault_schedule.append({
            "round": r,
            "control": c,
            "target": t,
            "error_wires": ew,
            "error_types": et,
        })

    return fault_schedule

In [ ]:
n_rounds = 4
shots = 1

get_syndrome_function = make_fault_schedule_repeated_syndrome_qnode(
    n_rounds=n_rounds,
    shots=shots,
)

fault_schedule = make_fault_schedule(
    controls=[0] * n_rounds,
    targets=[9] * n_rounds,
    error_wires_list=[[0, 9]] * n_rounds,
    error_types_list=[["X", "Z"]] * n_rounds,
    error_rounds=list(range(n_rounds)),
)

raw = get_syndrome_function(fault_schedule=fault_schedule)

syndrome = reshape_raw_syndrome(raw, n_rounds, shots)
detection_events = compute_detection_events(syndrome)

print_single_shot_sequence(
    syndrome,
    detection_events,
    shot_idx=0,
)

Syndrome sequence
-----------------
round 0: Z=[0 0 0 0]  X=[1 0 0 0]
round 1: Z=[1 0 0 0]  X=[0 0 0 0]
round 2: Z=[0 0 0 0]  X=[1 0 0 0]
round 3: Z=[1 0 0 0]  X=[0 0 0 0]

Detection events
----------------
between round 0 and 1: Z=[1 0 0 0]  X=[1 0 0 0]
between round 1 and 2: Z=[1 0 0 0]  X=[1 0 0 0]
between round 2 and 3: Z=[1 0 0 0]  X=[1 0 0 0]


/Users/jhan/Library/Mobile Documents/com~apple~CloudDocs/ETRI/연구/Correlated Noise Estimation by Syndrome Measurement/syndrome_env/lib/python3.14/site-packages/pennylane/devices/device_api.py:201: PennyLaneDeprecationWarning: Setting shots on device is deprecated. Please use the `set_shots` transform on the respective QNode instead.
  warnings.warn(


In [50]:
qml.draw_mpl(get_syndrome_function)(fault_schedule=fault_schedule)

(<Figure size 20300x2650 with 1 Axes>, <Axes: >)

In [51]:
def enumerate_stabilizer_cnots():
    """
    현재 stabilizer measurement circuit에서 등장하는 모든 directed CNOT 반환.

    Z stabilizer:
        data -> ancilla

    X stabilizer:
        ancilla -> data
    """

    cnot_locations = []

    for i, stab in enumerate(Z_stabilizers):
        anc = anc_z[i]
        for q in stab:
            cnot_locations.append({
                "stab_type": "Z",
                "stab_index": i,
                "control": q,
                "target": anc,
            })

    for i, stab in enumerate(X_stabilizers):
        anc = anc_x[i]
        for q in stab:
            cnot_locations.append({
                "stab_type": "X",
                "stab_index": i,
                "control": anc,
                "target": q,
            })

    return cnot_locations

In [52]:
loc = enumerate_stabilizer_cnots()[0]

fault_schedule = make_fault_schedule(
    controls=[loc["control"]] * n_rounds,
    targets=[loc["target"]] * n_rounds,
    error_wires_list=[[loc["control"], loc["target"]]] * n_rounds,
    error_types_list=[["X", "Z"]] * n_rounds,
    error_rounds=list(range(n_rounds)),
)

In [57]:
enumerate_stabilizer_cnots()

[{'stab_type': 'Z', 'stab_index': 0, 'control': 0, 'target': 9},
 {'stab_type': 'Z', 'stab_index': 0, 'control': 1, 'target': 9},
 {'stab_type': 'Z', 'stab_index': 0, 'control': 3, 'target': 9},
 {'stab_type': 'Z', 'stab_index': 0, 'control': 4, 'target': 9},
 {'stab_type': 'Z', 'stab_index': 1, 'control': 2, 'target': 10},
 {'stab_type': 'Z', 'stab_index': 1, 'control': 5, 'target': 10},
 {'stab_type': 'Z', 'stab_index': 2, 'control': 3, 'target': 11},
 {'stab_type': 'Z', 'stab_index': 2, 'control': 6, 'target': 11},
 {'stab_type': 'Z', 'stab_index': 3, 'control': 4, 'target': 12},
 {'stab_type': 'Z', 'stab_index': 3, 'control': 5, 'target': 12},
 {'stab_type': 'Z', 'stab_index': 3, 'control': 7, 'target': 12},
 {'stab_type': 'Z', 'stab_index': 3, 'control': 8, 'target': 12},
 {'stab_type': 'X', 'stab_index': 0, 'control': 13, 'target': 0},
 {'stab_type': 'X', 'stab_index': 0, 'control': 13, 'target': 1},
 {'stab_type': 'X', 'stab_index': 1, 'control': 14, 'target': 1},
 {'stab_type':

In [59]:
def make_single_cnot_fault_schedule(location, fault_round=0, error_types=["X", "Z"]):
    """
    특정 round의 특정 directed CNOT 직후에 single fault 하나만 삽입.

    location:
        {
            "stab_type": "Z" or "X",
            "stab_index": int,
            "control": int,
            "target": int,
        }

    error_types=["X", "Z"]이면:
        X on control, Z on target
    """

    control = location["control"]
    target = location["target"]

    return [
        {
            "round": fault_round,
            "control": control,
            "target": target,
            "error_wires": [control, target],
            "error_types": error_types,
        }
    ]

In [60]:
def generate_single_cnot_fault_dataset(
    get_syndrome_function,
    n_rounds,
    shots=1,
    fault_round=0,
    error_types=["X", "Z"],
    use_detection=True,
):
    """
    모든 directed CNOT location에 대해:
        single CNOT fault -> syndrome/detection pattern
    데이터를 생성.

    return:
        dataset: list of dict
    """

    cnot_locations = enumerate_stabilizer_cnots()
    dataset = []

    for cnot_index, loc in enumerate(cnot_locations):
        fault_schedule = make_single_cnot_fault_schedule(
            location=loc,
            fault_round=fault_round,
            error_types=error_types,
        )

        raw = get_syndrome_function(
            fault_schedule=fault_schedule
        )

        syndrome = reshape_raw_syndrome(
            raw,
            n_rounds=n_rounds,
            shots=shots,
        )

        detection_events = compute_detection_events(syndrome)

        if use_detection:
            x = detection_events[0].astype(int)
        else:
            x = syndrome[0].astype(int)

        dataset.append({
            "label": cnot_index,
            "cnot_location": loc,
            "fault_schedule": fault_schedule,
            "syndrome": syndrome[0].astype(int),
            "detection_events": detection_events[0].astype(int),
            "x": x,
        })

    return dataset

In [64]:
n_rounds = 4
shots = 1

get_syndrome_function = make_fault_schedule_repeated_syndrome_qnode(
    n_rounds=n_rounds,
    shots=shots,
)

dataset = generate_single_cnot_fault_dataset(
    get_syndrome_function=get_syndrome_function,
    n_rounds=n_rounds,
    shots=shots,
    fault_round=0,
    error_types=["X", "Z"],
    use_detection=False,
)

print("number of samples:", len(dataset))

for item in dataset:
    print()
    print("label:", item["label"])
    print("location:", item["cnot_location"])
    print("syndrome:")
    print(item["syndrome"])
    print("detection:")
    print(item["detection_events"])

number of samples: 24

label: 0
location: {'stab_type': 'Z', 'stab_index': 0, 'control': 0, 'target': 9}
syndrome:
[[0 0 0 0 1 0 0 0]
 [1 0 0 0 1 0 0 0]
 [1 0 0 0 1 0 0 0]
 [1 0 0 0 1 0 0 0]]
detection:
[[1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]

label: 1
location: {'stab_type': 'Z', 'stab_index': 0, 'control': 1, 'target': 9}
syndrome:
[[0 0 0 0 0 1 0 0]
 [1 0 0 0 0 1 0 0]
 [1 0 0 0 0 1 0 0]
 [1 0 0 0 0 1 0 0]]
detection:
[[1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]

label: 2
location: {'stab_type': 'Z', 'stab_index': 0, 'control': 3, 'target': 9}
syndrome:
[[0 0 1 0 0 1 1 0]
 [1 0 1 0 0 1 1 0]
 [1 0 1 0 0 1 1 0]
 [1 0 1 0 0 1 1 0]]
detection:
[[1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]

label: 3
location: {'stab_type': 'Z', 'stab_index': 0, 'control': 4, 'target': 9}
syndrome:
[[0 0 0 1 0 0 0 0]
 [1 0 0 1 0 0 0 0]
 [1 0 0 1 0 0 0 0]
 [1 0 0 1 0 0 0 0]]
detection:
[[1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]

label: 4
location: {'

In [65]:
def make_fixed_cnot_fault_schedule(
    control,
    target,
    error_rounds,
    error_types=["X", "Z"],
):
    """
    고정된 directed CNOT(control -> target)에 대해,
    지정된 round들에서만 fault를 삽입.

    error_types=["X", "Z"]이면:
        X on control, Z on target

    예:
        control=0
        target=9
        error_rounds=[0, 2]

    의미:
        round 0의 CNOT(0 -> 9) 직후 X_0 Z_9
        round 2의 CNOT(0 -> 9) 직후 X_0 Z_9
    """

    fault_schedule = []

    for r in error_rounds:
        fault_schedule.append({
            "round": r,
            "control": control,
            "target": target,
            "error_wires": [control, target],
            "error_types": error_types,
        })

    return fault_schedule

In [66]:
def generate_single_occurrence_dataset_for_fixed_cnot(
    get_syndrome_function,
    n_rounds,
    shots,
    control,
    target,
    error_types=["X", "Z"],
):
    """
    고정된 CNOT(control -> target)에 대해,
    전체 repeated syndrome cycle 중 fault가 딱 한 round에서만 발생하는 경우들을 생성.

    samples:
        round 0 only
        round 1 only
        ...
        round n_rounds-1 only
    """

    dataset = []

    for fault_round in range(n_rounds):
        fault_schedule = make_fixed_cnot_fault_schedule(
            control=control,
            target=target,
            error_rounds=[fault_round],
            error_types=error_types,
        )

        raw = get_syndrome_function(fault_schedule=fault_schedule)

        syndrome = reshape_raw_syndrome(
            raw,
            n_rounds=n_rounds,
            shots=shots,
        )

        detection_events = compute_detection_events(syndrome)

        dataset.append({
            "label": fault_round,
            "control": control,
            "target": target,
            "error_rounds": [fault_round],
            "fault_schedule": fault_schedule,
            "syndrome": syndrome[0].astype(int),
            "detection_events": detection_events[0].astype(int),
        })

    return dataset

In [67]:
n_rounds = 4
shots = 1

get_syndrome_function = make_fault_schedule_repeated_syndrome_qnode(
    n_rounds=n_rounds,
    shots=shots,
)

single_occurrence_dataset = generate_single_occurrence_dataset_for_fixed_cnot(
    get_syndrome_function=get_syndrome_function,
    n_rounds=n_rounds,
    shots=shots,
    control=0,
    target=9,
    error_types=["X", "Z"],
)

for item in single_occurrence_dataset:
    print()
    print("fault round:", item["label"])
    print("syndrome:")
    print(item["syndrome"])
    print("detection events:")
    print(item["detection_events"])


fault round: 0
syndrome:
[[0 0 0 0 1 0 0 0]
 [1 0 0 0 1 0 0 0]
 [1 0 0 0 1 0 0 0]
 [1 0 0 0 1 0 0 0]]
detection events:
[[1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]

fault round: 1
syndrome:
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0]
 [1 0 0 0 1 0 0 0]
 [1 0 0 0 1 0 0 0]]
detection events:
[[0 0 0 0 1 0 0 0]
 [1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]

fault round: 2
syndrome:
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0]
 [1 0 0 0 1 0 0 0]]
detection events:
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0]
 [1 0 0 0 0 0 0 0]]

fault round: 3
syndrome:
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0]]
detection events:
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0]]


In [68]:
from itertools import combinations

def all_error_round_subsets(n_rounds, include_empty=True):
    """
    n_rounds에 대해 가능한 모든 error round subset 생성.

    예: n_rounds=3
        []
        [0]
        [1]
        [2]
        [0,1]
        [0,2]
        [1,2]
        [0,1,2]
    """

    subsets = []

    start_k = 0 if include_empty else 1

    for k in range(start_k, n_rounds + 1):
        for comb in combinations(range(n_rounds), k):
            subsets.append(list(comb))

    return subsets

In [69]:
def generate_all_occurrence_patterns_for_fixed_cnot(
    get_syndrome_function,
    n_rounds,
    shots,
    control,
    target,
    error_types=["X", "Z"],
    include_empty=True,
):
    """
    고정된 CNOT(control -> target)에 대해,
    모든 round-wise fault occurrence pattern을 생성.

    sample 수:
        include_empty=True  -> 2^n_rounds
        include_empty=False -> 2^n_rounds - 1
    """

    dataset = []

    error_round_patterns = all_error_round_subsets(
        n_rounds=n_rounds,
        include_empty=include_empty,
    )

    for pattern_id, error_rounds in enumerate(error_round_patterns):
        fault_schedule = make_fixed_cnot_fault_schedule(
            control=control,
            target=target,
            error_rounds=error_rounds,
            error_types=error_types,
        )

        raw = get_syndrome_function(fault_schedule=fault_schedule)

        syndrome = reshape_raw_syndrome(
            raw,
            n_rounds=n_rounds,
            shots=shots,
        )

        detection_events = compute_detection_events(syndrome)

        occurrence_vector = np.zeros(n_rounds, dtype=int)
        occurrence_vector[error_rounds] = 1

        dataset.append({
            "label": pattern_id,
            "occurrence_vector": occurrence_vector,
            "control": control,
            "target": target,
            "error_rounds": error_rounds,
            "fault_schedule": fault_schedule,
            "syndrome": syndrome[0].astype(int),
            "detection_events": detection_events[0].astype(int),
        })

    return dataset

In [70]:
n_rounds = 4
shots = 1

get_syndrome_function = make_fault_schedule_repeated_syndrome_qnode(
    n_rounds=n_rounds,
    shots=shots,
)

dataset = generate_all_occurrence_patterns_for_fixed_cnot(
    get_syndrome_function=get_syndrome_function,
    n_rounds=n_rounds,
    shots=shots,
    control=0,
    target=9,
    error_types=["X", "Z"],
    include_empty=True,
)

print("number of samples:", len(dataset))
# expected: 2^n_rounds = 16

for item in dataset:
    print()
    print("occurrence:", item["occurrence_vector"])
    print("error rounds:", item["error_rounds"])
    print("syndrome:")
    print(item["syndrome"])
    print("detection:")
    print(item["detection_events"])

number of samples: 16

occurrence: [0 0 0 0]
error rounds: []
syndrome:
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]
detection:
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]

occurrence: [1 0 0 0]
error rounds: [0]
syndrome:
[[0 0 0 0 1 0 0 0]
 [1 0 0 0 1 0 0 0]
 [1 0 0 0 1 0 0 0]
 [1 0 0 0 1 0 0 0]]
detection:
[[1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]

occurrence: [0 1 0 0]
error rounds: [1]
syndrome:
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0]
 [1 0 0 0 1 0 0 0]
 [1 0 0 0 1 0 0 0]]
detection:
[[0 0 0 0 1 0 0 0]
 [1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]

occurrence: [0 0 1 0]
error rounds: [2]
syndrome:
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0]
 [1 0 0 0 1 0 0 0]]
detection:
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0]
 [1 0 0 0 0 0 0 0]]

occurrence: [0 0 0 1]
error rounds: [3]
syndrome:
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0]]
detection:
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 

In [71]:
from collections import defaultdict

def pattern_key(array):
    return tuple(array.astype(int).flatten())


def find_duplicate_patterns(dataset, key_name="syndrome"):
    pattern_to_items = defaultdict(list)

    for item in dataset:
        key = pattern_key(item[key_name])
        pattern_to_items[key].append(item)

    duplicates = {
        key: items
        for key, items in pattern_to_items.items()
        if len(items) > 1
    }

    return pattern_to_items, duplicates

In [72]:
pattern_to_items, duplicates = find_duplicate_patterns(
    dataset,
    key_name="syndrome",
)

print("number of occurrence patterns:", len(dataset))
print("number of unique syndrome patterns:", len(pattern_to_items))
print("number of duplicated syndrome patterns:", len(duplicates))

for key, items in duplicates.items():
    print()
    print("duplicate syndrome pattern:")
    print(np.array(key).reshape(n_rounds, 8))

    print("occurrence vectors:")
    for item in items:
        print(item["occurrence_vector"], "error_rounds =", item["error_rounds"])

number of occurrence patterns: 16
number of unique syndrome patterns: 16
number of duplicated syndrome patterns: 0
